# Life2Lang — Fine-tuning

Fine-tune the pretrained `khairi/life2lang-base-pt` model on the multi-task
protein instruction dataset (`khairi/life2lang-instruction-dataset`).

Tasks covered by the dataset:
| Aspect | Example prefix |
|--------|----------------|
| GO-BP  | predict biological processes from protein sequence: |
| GO-MF  | predict molecular functions from protein sequence: |
| GO-CC  | predict protein localization sites: |
| Family | predict protein family from sequence: / design protein sequence for family: |
| Function description | predict catalytic activity from protein sequence: |

**Outputs**
- Experiment tracked on [Weights & Biases](https://wandb.ai)
- Final model pushed to 🤗 Hub as `khairi/life2lang-base-ft`

**Runtime**: GPU required.

## 1 · Install

In [ ]:
!pip install -q git+https://github.com/abidikhairi/life2lang.git
!pip install -q wandb millify

## 2 · Authenticate

In [ ]:
import wandb
wandb.login()

In [ ]:
from huggingface_hub import login
login()  # paste a token with write access to khairi/life2lang-base-ft

## 3 · Configuration

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────
DATASET_ID = "khairi/life2lang-instruction-dataset"
# Set to a specific aspect to train on a single task, or None for all tasks.
# Valid aspects: 'GO-BP', 'GO-MF', 'GO-CC', 'family', 'function description',
#                'catalyctic activity'
FILTER_ASPECT = None

# ── Model ─────────────────────────────────────────────────────────────────
BASE_MODEL   = "khairi/life2lang-base-pt"
HUB_MODEL_ID = "khairi/life2lang-base-ft"
OUTPUT_DIR   = "/tmp/life2lang-base-ft"

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE           = 4
EVAL_BATCH_SIZE      = 8
GRADIENT_ACCUM_STEPS = 4    # effective batch = 16
NUM_EPOCHS           = 5
LEARNING_RATE        = 5e-4
WEIGHT_DECAY         = 0.01
MAX_GRAD_NORM        = 0.1
WARMUP_RATIO         = 0.1
LR_SCHEDULER         = "cosine"
EVAL_STEPS           = 200
SAVE_STEPS           = 200
LOGGING_STEPS        = 50

# ── W&B ───────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "life2lang"
WANDB_RUN_NAME = "finetuning-base"

## 4 · Environment check

In [ ]:
import os
from millify import millify

from life2lang.utils import (
    print_gpu_info,
    print_gpu_memory,
    clean_gpu_memory,
    count_trainable_parameters,
)

print_gpu_info()
print_gpu_memory()

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

## 5 · Load dataset

In [ ]:
from datasets import load_dataset

dataset   = load_dataset(DATASET_ID)
train_raw = dataset["train"]
valid_raw = dataset["test"]

if FILTER_ASPECT is not None:
    train_raw = train_raw.filter(lambda x: x["aspect"] == FILTER_ASPECT)
    valid_raw = valid_raw.filter(lambda x: x["aspect"] == FILTER_ASPECT)

print(f"Train size : {len(train_raw):,}")
print(f"Valid size : {len(valid_raw):,}")
train_raw[0]

## 6 · Tokeniser & data collator

The dataset is kept as raw text. A custom `ProteinCollator` tokenises
and pads each mini-batch on the fly, so no pre-tokenised copies are
stored on disk. Each input is formatted as:

```
{prefix}
{input}          ← protein sequence
```

Label padding tokens are replaced with `-100` so they are ignored by
the cross-entropy loss.

In [ ]:
from life2lang.models import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {tokenizer.vocab_size}")

In [ ]:
import torch
from dataclasses import dataclass, field
from typing import Any, Dict, List


@dataclass
class ProteinCollator:
    """Tokenises and pads a raw-text batch on the fly."""

    tokenizer: Any
    max_input_length: int = 512
    max_target_length: int = 128
    label_pad_token_id: int = -100

    def __call__(self, examples: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_texts  = [f"{ex['prefix']}\n{ex['input']}" for ex in examples]
        target_texts = [ex["target"] for ex in examples]

        model_inputs = self.tokenizer(
            input_texts,
            padding=True,
            truncation=True,
            max_length=self.max_input_length,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            target_texts,
            padding=True,
            truncation=True,
            max_length=self.max_target_length,
            return_tensors="pt",
        )

        label_ids = labels["input_ids"].clone()
        label_ids[label_ids == self.tokenizer.pad_token_id] = self.label_pad_token_id
        model_inputs["labels"] = label_ids
        return model_inputs


data_collator = ProteinCollator(tokenizer)

## 7 · Model

In [ ]:
from life2lang.models import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

n_params = count_trainable_parameters(model)
print(f"Trainable parameters : {millify(n_params)} ({n_params:,})")
print(f"Max context distance : {model.config.relative_attention_max_distance}")

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batching
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM_STEPS,

    # Schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,

    # Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,

    # Logging
    report_to="wandb",
    run_name=WANDB_RUN_NAME,

    # Hub
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="checkpoint",

    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_raw,
    eval_dataset=valid_raw,
    data_collator=data_collator,
)

clean_gpu_memory()
trainer.train()

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batching
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM_STEPS,

    # Schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,

    # Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,

    # Logging
    report_to="wandb",
    run_name=WANDB_RUN_NAME,

    # Hub
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="checkpoint",

    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_raw,
    eval_dataset=valid_raw,
    data_collator=data_collator,
)

clean_gpu_memory()
trainer.train()

## 9 · Save & push

In [ ]:
final_dir = OUTPUT_DIR + "/final_model"

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Saved locally → {final_dir}")

trainer.push_to_hub(commit_message="final fine-tuned model")
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"Pushed → https://huggingface.co/{HUB_MODEL_ID}")

In [ ]:
wandb.finish()